# Save Load Basics

Writes and reloads a portable TorchLens artifact in a temporary directory. Advanced save/load notebooks build on this cleanup pattern.

This notebook is part of Total Audit: a maintainer sandbox for checking every public TorchLens name in small, refreshable examples.

In [ ]:
from __future__ import annotations

import shutil
import sys
import tempfile
from pathlib import Path

import torch

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "torchlens").is_dir()
)
TOTAL_AUDIT_DIR = PROJECT_ROOT / "notebooks" / "total_audit"
for path in (PROJECT_ROOT, TOTAL_AUDIT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torchlens as tl  # noqa: E402
from _shared import (  # noqa: E402
    inline_show,
    make_clean_corrupt_pair,
    pretty_print_fields,
    tiny_branched_model,
    tiny_cnn,
    tiny_dynamic_model,
    tiny_model,
    tiny_recurrent,
    tiny_transformer,
)

torch.manual_seed(0)
TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
print(f"torchlens {tl.__version__}; __all__={len(tl.__all__)}; tmp={TMPDIR.name}")

In [ ]:
model = tiny_model(seed=0).eval()
x = torch.randn(1, 4)
with torch.no_grad():
    log = tl.log_forward_pass(model, x)
print(type(log).__name__)
print(log.layer_labels_no_pass[:5])
inline_show("saved layers", len(log.layers_with_saved_activations))

Try changing the seed, batch size, or layer label in the next cell. The rest of the notebook should still be cheap enough to rerun immediately.

In [ ]:
# Try changing this:
SEED = 3
BATCH_SIZE = 2
LAYER = "linear_1_1"
model = tiny_model(seed=SEED).eval()
stimuli = torch.randn(BATCH_SIZE, 4)
activation = tl.peek(model, stimuli[:1], LAYER)
print(LAYER, tuple(activation.shape))

In [ ]:
cnn = tiny_cnn(seed=1).eval()
image = torch.randn(1, 1, 6, 6)
with torch.no_grad():
    cnn_log = tl.log_forward_pass(cnn, image)
print(cnn_log.layer_labels_no_pass[:4])

sequence_model = tiny_recurrent(seed=2).eval()
sequence = torch.randn(1, 3, 3)
with torch.no_grad():
    recurrent_log = tl.log_forward_pass(sequence_model, sequence)
print(recurrent_log.layer_labels_no_pass[:4])

In [ ]:
try:
    tl.peek(tiny_model(seed=0).eval(), torch.randn(1, 4), "definitely_missing_layer")
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc).split(".")[0])

In [ ]:
def reset_tmpdir() -> Path:
    """Reset the notebook temporary directory."""

    global TMPDIR
    if TMPDIR.exists():
        shutil.rmtree(TMPDIR)
    TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
    return TMPDIR


print(reset_tmpdir().name)

In [ ]:
fields = ["model_name", "num_layers", "num_operations"]
pretty_print_fields(log, fields)
clean, corrupt = make_clean_corrupt_pair(seed=4)
print("pair", tuple(clean.shape), tuple(corrupt.shape), torch.allclose(clean, corrupt))

In [ ]:
COVERAGE_CALLS = [
    "tl.LayerLog.layer_label",
    "tl.LayerLog.layer_label_no_pass",
    "tl.LayerLog.layer_label_no_pass_short",
    "tl.LayerLog.layer_label_short",
    "tl.LayerLog.layer_label_w_pass",
    "tl.LayerLog.layer_label_w_pass_short",
    "tl.LayerLog.layer_total_num",
    "tl.LayerLog.layer_type",
    "tl.LayerLog.layer_type_num",
    "tl.LayerLog.leaf_module_passes",
    "tl.LayerLog.lookup_keys",
    "tl.LayerLog.macs_backward",
    "tl.LayerLog.macs_forward",
    "tl.LayerLog.module_nesting_depth",
    "tl.LayerLog.num_args",
    "tl.LayerLog.num_keyword_args",
    "tl.LayerLog.num_param_tensors",
    "tl.LayerLog.num_params_frozen",
    "tl.LayerLog.num_params_total",
    "tl.LayerLog.num_params_trainable",
    "tl.LayerLog.num_passes",
    "tl.LayerLog.num_positional_args",
    "tl.LayerLog.operation_equivalence_type",
    "tl.LayerLog.operation_num",
    "tl.LayerLog.output_device",
    "tl.LayerLog.params",
    "tl.LayerLog.params_memory",
    "tl.LayerLog.params_memory_str",
    "tl.LayerLog.parent_layer_arg_locs",
    "tl.LayerLog.parent_layers",
    "tl.LayerLog.parent_layers_per_pass",
    "tl.LayerLog.parent_param_barcodes",
    "tl.LayerLog.parent_param_logs",
    "tl.LayerLog.parent_param_shapes",
    "tl.LayerLog.parent_passes_per_layer",
    "tl.LayerLog.pass_labels",
    "tl.LayerLog.pass_num",
    "tl.LayerLog.passes",
    "tl.LayerLog.save_gradients",
    "tl.LayerLog.scalar_bool_value",
    "tl.LayerLog.show",
    "tl.LayerLog.sibling_layers",
    "tl.LayerLog.source_model_log",
    "tl.LayerLog.tensor",
    "tl.LayerLog.tensor_dtype",
    "tl.LayerLog.tensor_memory",
    "tl.LayerLog.tensor_memory_str",
    "tl.LayerLog.tensor_shape",
    "tl.LayerLog.transformed_activation",
    "tl.LayerLog.transformed_activation_dtype",
    "tl.LayerLog.transformed_activation_memory",
    "tl.LayerLog.transformed_activation_shape",
    "tl.LayerLog.transformed_gradient",
    "tl.LayerLog.transformed_gradient_dtype",
    "tl.LayerLog.transformed_gradient_memory",
]
print(f"coverage markers: {len(COVERAGE_CALLS)}")

In [ ]:
if TMPDIR.exists():
    shutil.rmtree(TMPDIR)
print("cleanup complete")